### Fixing the environment

In [1]:
#Importing packages
import sys
sys.path.append('/home/onyxia/work/nlp_central_banks/lyna_work')

from module import *

/home/onyxia/work/venv_gpu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Importing torch and making sure the GPU is available
import sys
print(sys.executable)  # doit afficher .../venv_gpu/bin/python

import torch
print(torch.cuda.is_available())  # doit afficher True
print(torch.cuda.get_device_name(0))  # NVIDIA A2

/home/onyxia/work/venv_gpu/bin/python
True
Tesla T4


In [3]:
# ── Verification GPU ────────────────────────────────────
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch  : 2.6.0+cu124
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


### BERTopic pipeline
#### Preprocessing the data

I import my preprocessed data.

In [4]:
# Importing the preprocessed data from the first notebook (saved in the SSP Cloud) :
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)
fs.invalidate_cache()
with fs.open("lelkamel/chunks/df_filtered.csv", "rb") as f:
    df_filtered = pd.read_csv(f)
print(f"df_filtered chargé : {df_filtered.shape}")

df_filtered chargé : (2286, 19)


Now I will compare my text's size to that of the model that interests me, to see whether it fits the context window.

I begin by retrieving the context window size of the model that I will use on the HuggingFace.

In [11]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("hugom123/bge-centralbank")
print(config.max_position_embeddings)

512


In [12]:
df_filtered["text.len"] = df_filtered["text"].apply(len)
df_filtered["text.len"].describe()

count      2286.000000
mean      20002.870954
std       11961.596277
min        1314.000000
25%       11670.250000
50%       17653.500000
75%       25741.750000
max      122759.000000
Name: text.len, dtype: float64

My text clearly does not fit the context window of this model. To resolve this, I decide to chunk it. This strategy will also enable me to retrive topics within speeches. 

In [ ]:
# I do the chunking, thanks to a function I built, available in the module section
#if torch.cuda.is_available():
#    spacy.prefer_gpu()  # utilise GPU si CuPy dispo, sinon CPU sans erreur

#nlp = spacy.load("en_core_web_sm")

#tokenizer = AutoTokenizer.from_pretrained("hugom123/bge-centralbank")

#df_chunks = process_corpus(df_filtered)

#print(f"\nDiscours        : {len(df_filtered)}")
#print(f"Chunks          : {len(df_chunks)}")
#print(f"Chunks/discours : {len(df_chunks)/len(df_filtered):.1f}")
#print(f"\nTaille chunks (mots) :")
#print(df_chunks['n_words'].describe().round(0))
#print(f"\nChunks < 50 mots : {(df_chunks['n_words'] < 50).sum()}")
#I save it 
#df_chunks.to_csv("s3://lelkamel/chunks/df_chunks_all.csv", index=False)


Segmentation en phrases (spaCy)...


spaCy: 100%|██████████| 2286/2286 [13:53<00:00,  2.74it/s]


Chunking...


Chunking: 100%|██████████| 2286/2286 [00:26<00:00, 86.06it/s] 



Discours        : 2286
Chunks          : 23913
Chunks/discours : 10.5

Taille chunks (mots) :
count    23913.0
mean       342.0
std         58.0
min         18.0
25%        339.0
50%        359.0
75%        372.0
max        477.0
Name: n_words, dtype: float64

Chunks < 50 mots : 42


In [5]:
#To save computing time, I import here the chunk database, obtained and saved in the previous code cell.
with fs.open("lelkamel/chunks/df_chunks_all.csv", "rb") as f:
    df_chunks_all = pd.read_csv(f)

print(df_chunks_all.columns.tolist())
print(f"\nNombre de discours uniques : {df_chunks_all['doc_id'].nunique()}")
print(f"Chunks par discours (moy.) : {len(df_chunks_all)/df_chunks_all['doc_id'].nunique():.1f}")
print(f"\nAnnées : {df_chunks_all['Year'].min()} - {df_chunks_all['Year'].max()}")

['doc_id', 'chunk_id', 'CentralBank', 'Date', 'Year', 'Authorname', 'Role', 'Source', 'chunk_text', 'n_words']

Nombre de discours uniques : 2286
Chunks par discours (moy.) : 10.5

Années : 2001 - 2023


### I now create a BERTopic object, fit and transform

To create my `topic_model`, I start by creating a `BERTopic` object and define some parameters. 
- I begin by computing the embeddings of my chunks with `hugom123/bge-centralbank` and saving it in the onyxia cloud. 
- Secondly, I define my vectoriser model in order to remove common english stopwords as well as other economic and institutional stopwords to retrive meaningful topics. 
- Then, I do not change the default parameters of the clutering model at start (`hdbscan_model`), nor the dimension reduction model (`umap_model`). 
- Finally, I use the fit methode to fit the topic model to the corpus.

In [ ]:
# In this cell, I compute the embedding for all chunks
 #I begin by importing chunks, then import the model, then apply it on my data, then save it.   


#docs = df_chunks_all['chunk_text'].tolist()
#print(f"Nombre de chunks : {len(docs)}")
#print("\nChargement du modèle...")
#embedding_model = SentenceTransformer("hugom123/bge-centralbank")
#print("Calcul des embeddings...")
#embeddings = embedding_model.encode(
#    docs,
#    batch_size=64,
#    show_progress_bar=True,
#    device="cuda" if torch.cuda.is_available() else "cpu"
#)
#print(f"Embeddings shape : {embeddings.shape}")
#print("\nSauvegarde embeddings sur S3...")
#with fs.open("lelkamel/chunks/embeddings_all.npy", "wb") as f:
#    np.save(f, embeddings)
#print("Embeddings sauvegardés : s3://lelkamel/chunks/embeddings_all.npy")

Fichiers disponibles :
['lelkamel/chunks/df_all_clean.csv', 'lelkamel/chunks/df_chunks.csv', 'lelkamel/chunks/df_chunks_all.csv', 'lelkamel/chunks/df_stratified_clean.csv', 'lelkamel/chunks/embeddings.npy']

Chargement df_chunks...
df_chunks chargé : (23913, 10)
Nombre de chunks : 23913

Chargement du modèle...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2917.10it/s]


Calcul des embeddings...


Batches: 100%|██████████| 374/374 [33:56<00:00,  5.45s/it]


Embeddings shape : (23913, 1024)

Sauvegarde embeddings sur S3...
Embeddings sauvegardés : s3://lelkamel/chunks/embeddings_all.npy


In [6]:
#To save computing time, I import here the embeddings obtained and saved in the previous code cell.
with fs.open("lelkamel/chunks/embeddings_all.npy", "rb") as f:
    embeddings = np.load(f)

print(f"Embeddings chargés : {embeddings.shape}")

Embeddings chargés : (23913, 1024)


In [5]:
vectorizer_model = CountVectorizer(
    stop_words= get_stopwords(),
    ngram_range=(1, 2),
    min_df=2,
    lowercase=True
)

In the next cell, I run the Bertopic model. I finetune it so that I control more the granularity of the model. I select `n_neighbors` and `min_cluster_size` to be equal to 50. This enables me to get 108 topics with enough granularity. For example, I have two climate topics, one which focuses on financial risks from climate, the other focuses on green transition and ESG. These are not exactly the same and it is interesting to see these differences (see below for this example).

In [ ]:
#docs = df_chunks_all['chunk_text'].tolist()

#RANDOM_SEED = 42
#default_umap_parameters = {
#    "n_neighbors": 50,
#    "n_components": 5,
#    "min_dist": 0.0,
#    "metric": "cosine",
#}
#hdbscan_model = HDBSCAN(
#    min_cluster_size=50,      # plus petit = plus de topics
#    min_samples=5,            # moins strict = moins d'outliers
#    metric='euclidean',
#    cluster_selection_method='leaf',  # 'leaf' crée plus de petits clusters
#    prediction_data=True
#)

#topic_model = BERTopic(
#    language="english",
#    embedding_model="hugom123/bge-centralbank",
#    vectorizer_model=vectorizer_model,
#    umap_model=UMAP(**default_umap_parameters, random_state=42),
#    hdbscan_model=hdbscan_model,
#    min_topic_size=20,
#   nr_topics=None,
#    calculate_probabilities=True,
#   verbose=True,
#)

#topics, probs = topic_model.fit_transform(
#    documents=docs,
#    embeddings=embeddings
#)

#df_chunks_all['topic']      = topics
#df_chunks_all['topic_prob'] = probs.max(axis=1) if probs.ndim > 1 else probs

#print(f"\nNombre de topics : {topic_model.get_topic_info().shape[0] - 1}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2574.13it/s]
2026-04-29 11:04:46,255 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-29 11:05:57,048 - BERTopic - Dimensionality - Completed ✓
2026-04-29 11:05:57,052 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-29 11:06:06,774 - BERTopic - Cluster - Completed ✓
2026-04-29 11:06:06,787 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-29 11:06:16,819 - BERTopic - Representation - Completed ✓



Nombre de topics : 109


In [9]:
topics, probabilities = topic_model.transform(documents=docs, embeddings=embeddings)

2026-04-29 11:10:10,953 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-29 11:10:11,132 - BERTopic - Dimensionality - Completed ✓
2026-04-29 11:10:11,133 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-29 11:10:11,780 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-29 11:10:20,886 - BERTopic - Probabilities - Completed ✓
2026-04-29 11:10:20,890 - BERTopic - Cluster - Completed ✓


In [10]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,10578,-1_credit_term_risks_us,"[credit, term, risks, us, see, prices, banking...","[Fortunately, this scenario has so far not com..."
1,0,416,0_liquidity_operations_refinancing_refinancing...,"[liquidity, operations, refinancing, refinanci...",[We have decided on three-year refinancing ope...
2,1,409,1_fiscal_pact_countries_public,"[fiscal, pact, countries, public, union, struc...","[Nonetheless, further reforms in the fiscal ar..."
3,2,362,2_supervisory_supervision_eu_banking,"[supervisory, supervision, eu, banking, ssm, e...","[In addition, a new idea of reform contracts w..."
4,3,359,3_medium_medium term_outlook_term,"[medium, medium term, outlook, term, developme...","[Turning to the price developments, annual HIC..."
...,...,...,...,...,...
105,104,52,104_flows_emerging_emes_inflows,"[flows, emerging, emes, inflows, push, pull fa...",[II. Growing challenges in the Current IMFS Ch...
106,105,52,105_productivity_1995_per_output,"[productivity, 1995, per, output, labor produc...",[Different methods of data construction will y...
107,106,52,106_loans_liquidity_lending_credit,"[loans, liquidity, lending, credit, commercial...",[BIS central bankers speeches 3 recognition th...
108,107,51,107_spending_housing_sales_construction,"[spending, housing, sales, construction, house...","[Hence, with production running well below sal..."


The BERTopic model identifies **109 topics** from the corpus of 30,401 chunks. Several of them are immediately interpretable and consistent with what one would expect from central bank communication over the 2001--2023 period:

- **Topic 15 — Housing & Mortgages** (`mortgage, mortgages, subprime, borrowers`):  not surprising given that the 2008 financial crisis shed light on the interaction between the housing market and financial stability; we will examine whether this topic is predominantly concentrated in the post-2008 period.

- **Topic 41 — Digital Money & Payments** (`digital, money, payments, payment`): distinct from the crypto topic, this cluster captures the debate around Central Bank Digital Currencies (CBDCs) and the modernisation of payment systems.

- **Topic 38 — Inequality & Wealth** (`wealth, income, inequality, families`): a particularly interesting topic, as distributional concerns are not historically part of the core central bank mandate. Its presence in the corpus warrants further investigation into its temporal distribution.

- **Topics 77 & 94 — Climate Change** (`climate change, risks, green, transition`): consistent with the findings of Campiglio et al. (2025); BERTopic identifies two distinct climate-related clusters — one centred on climate as a **financial risk** (stress testing, TCFD disclosures) and one on the **green transition** agenda (ESG, net zero, carbon) — which confirms and refines their dictionary-based approach.

- **Topic 90 — Crypto & Stablecoins** (`crypto, stablecoins, stablecoin, bitcoin`): an emerging topic reflecting the growing attention of central bankers to decentralised finance and the regulatory challenges it poses.

- **Topic 91 — 2008 Financial Crisis** (`aig, credit, treasury, institutions`): this topic clearly captures the acute phase of the 2008 crisis, referencing key institutional actors (AIG, Treasury) and emergency interventions.

However, **Topic -1 accounts for 10,578 chunks, representing 34.8% of the corpus** — a substantial share of documents that HDBSCAN was unable to assign to any cluster. This is a known limitation of density-based clustering: documents that do not belong to a sufficiently dense region of the embedding space are labelled as outliers. 

To address this, I apply the `reduce_outliers` method with the **embedding strategy**. This approach assigns each outlier chunk to the topic whose centroid is closest in the embedding space, using cosine similarity. 

In [ ]:
new_topics = topic_model.reduce_outliers(
    documents=docs,
    topics=topics,
    probabilities=probs,
    embeddings=embeddings,
    strategy="embeddings",
    threshold=0.0
)

# Mettre à jour le modèle
topic_model.update_topics(
    docs,
    topics=new_topics,
    vectorizer_model=vectorizer_model
)

df_chunks_all['topic'] = new_topics
print(f"Outliers restants : {(pd.Series(new_topics) == -1).sum()}")
print("\n=== Top 30 topics ===")
print(topic_model.get_topic_info().head(30))
print(topic_model.get_topic_info().head(30))

with fs.open("lelkamel/chunks/df_chunks_with_topics.csv", "wb") as f:
    df_chunks_all.to_csv(f, index=False)
print("df_chunks_with_topics sauvegardé sur S3 !")

I don't have any outlier anymore ! See the next notebook for more visualisation.